# Synthetic geometry stress test

This manuscript-oriented notebook visualizes one frozen held-out synthetic job across a reference representation, an equivalent rotated representation, a diagnostically distorted representation, and an explicitly corrupted representation.

The notebook does not rerun the manuscript simulation suite, does not change ScGeo methods, does not tune thresholds, and does not alter truth definitions. It reads the frozen per-job audit outputs and deterministically regenerates only the selected synthetic cell geometry from the frozen job config so cells can be plotted.

In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata as importlib_metadata
import importlib.util
import json
import os
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path("/tmp") / "scgeo_revision_matplotlib"))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 220,
    "font.size": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.16,
})

STATE_COLORS = {
    "state_0": "#4C78A8",
    "state_1": "#F58518",
    "state_2": "#54A24B",
    "state_3": "#B279A2",
    "state_4": "#E45756",
}
CONDITION_MARKERS = {"control": "o", "treated": "^"}
SELECTED_JOB_ID = "manuscript_representation_corruption_evaluation_seed5"
SELECTED_REPS = [
    {"rep": "X_truth", "display": "Reference representation", "role": "reference"},
    {"rep": "X_rotated", "display": "Equivalent rotated/scaled family", "role": "equivalent"},
    {"rep": "X_anisotropic", "display": "Diagnostically distorted representation", "role": "diagnostic_distortion"},
    {"rep": "X_corrupted", "display": "Explicitly corrupted representation", "role": "explicit_corruption"},
]


def find_repo_root(start: Path | None = None) -> Path:
    path = (start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "configs" / "manuscript_benchmark_v1.json").exists():
            return candidate
    raise RuntimeError("Could not locate repository root from current working directory")


REPO_ROOT = find_repo_root()
CONFIG = json.loads((REPO_ROOT / "configs" / "manuscript_benchmark_v1.json").read_text(encoding="utf-8"))


def resolve_config_path(value: str) -> Path:
    path = Path(value).expanduser()
    if not path.is_absolute():
        path = REPO_ROOT / path
    return path.resolve()


BENCHMARK_DIR = resolve_config_path(os.environ.get(CONFIG["benchmark_dir_env"], CONFIG["default_benchmark_dir"]))
SOURCE_REPO = resolve_config_path(os.environ.get(CONFIG["source_repository_env"], CONFIG["default_source_repository"]))
OUTPUT_DIR = resolve_config_path(os.environ.get(CONFIG["notebook_output_env"], CONFIG["default_output_dir"]))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def rel_display(path: Path) -> str:
    path = path.resolve()
    try:
        return path.relative_to(REPO_ROOT).as_posix()
    except ValueError:
        return path.as_posix()


def package_version(name: str) -> str:
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return "not-installed"


def git_commit(path: Path) -> str | None:
    if not (path / ".git").exists():
        return None
    try:
        return subprocess.check_output(["git", "-C", str(path), "rev-parse", "HEAD"], text=True).strip()
    except Exception:
        return None


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_selected_frozen_files(required_paths: list[str]) -> pd.DataFrame:
    manifest = pd.read_csv(REPO_ROOT / CONFIG["results_manifest"])
    manifest_idx = manifest.set_index("relative_path", drop=False)
    if set(manifest["protocol_version"].unique()) != {CONFIG["protocol_version"]}:
        raise AssertionError("Manifest protocol does not match config")
    if set(manifest["source_commit"].unique()) != {CONFIG["expected_source_commit"]}:
        raise AssertionError("Manifest source commit does not match config")
    missing = sorted(set(required_paths).difference(manifest_idx.index))
    if missing:
        raise AssertionError(f"Required selected-job files are absent from manifest: {missing}")
    rows = []
    for rel in required_paths:
        path = BENCHMARK_DIR / rel
        if not path.exists():
            raise FileNotFoundError(path)
        observed = sha256_file(path)
        expected = manifest_idx.loc[rel, "sha256"]
        if observed != expected:
            raise AssertionError(f"Checksum mismatch for {rel}")
        rows.append({"relative_path": rel, "sha256": observed, "size_bytes": int(path.stat().st_size)})
    source_commit = git_commit(SOURCE_REPO)
    if source_commit is not None and source_commit != CONFIG["expected_source_commit"]:
        raise AssertionError(f"Source repo commit mismatch: {source_commit}")
    return pd.DataFrame(rows)


def job_file(suffix: str) -> str:
    return f"{SELECTED_JOB_ID}_{suffix}"


def read_job_csv(suffix: str) -> pd.DataFrame:
    return pd.read_csv(BENCHMARK_DIR / job_file(suffix))


def load_simulation_module():
    module_path = SOURCE_REPO / "scgeo" / "bench" / "_simulation.py"
    spec = importlib.util.spec_from_file_location("scgeo_bench_simulation_frozen", module_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not load simulation module from {module_path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def write_figure_bundle(stem: str, fig: plt.Figure, source_table: pd.DataFrame, alt_text: str) -> pd.DataFrame:
    fig_dir = OUTPUT_DIR / "figures"
    source_dir = OUTPUT_DIR / "figure_sources"
    alt_dir = OUTPUT_DIR / "alt_text"
    for directory in (fig_dir, source_dir, alt_dir):
        directory.mkdir(parents=True, exist_ok=True)
    png_path = fig_dir / f"{stem}.png"
    svg_path = fig_dir / f"{stem}.svg"
    csv_path = source_dir / f"{stem}.csv"
    alt_path = alt_dir / f"{stem}.txt"
    source_table.to_csv(csv_path, index=False)
    fig.savefig(svg_path, format="svg", bbox_inches="tight", metadata={"Date": None})
    fig.savefig(png_path, format="png", bbox_inches="tight", metadata={"Software": "matplotlib"})
    alt_path.write_text(alt_text.strip() + "\n", encoding="utf-8")
    plt.close(fig)
    return pd.DataFrame([{ 
        "figure": stem,
        "png": rel_display(png_path),
        "svg": rel_display(svg_path),
        "source_csv": rel_display(csv_path),
        "alt_text": rel_display(alt_path),
    }])


def write_table_artifact(stem: str, table: pd.DataFrame, alt_text: str | None = None) -> pd.DataFrame:
    source_dir = OUTPUT_DIR / "figure_sources"
    source_dir.mkdir(parents=True, exist_ok=True)
    csv_path = source_dir / f"{stem}.csv"
    table.to_csv(csv_path, index=False)
    row = {"artifact": stem, "source_csv": rel_display(csv_path)}
    if alt_text is not None:
        alt_dir = OUTPUT_DIR / "alt_text"
        alt_dir.mkdir(parents=True, exist_ok=True)
        alt_path = alt_dir / f"{stem}.txt"
        alt_path.write_text(alt_text.strip() + "\n", encoding="utf-8")
        row["alt_text"] = rel_display(alt_path)
    return pd.DataFrame([row])


def write_metadata(stem: str, extra: dict[str, object] | None = None) -> pd.DataFrame:
    metadata_dir = OUTPUT_DIR / "metadata"
    metadata_dir.mkdir(parents=True, exist_ok=True)
    metadata = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "python": sys.version,
        "python_executable": sys.executable,
        "platform": platform.platform(),
        "packages": {name: package_version(name) for name in ["scgeo", "anndata", "pandas", "numpy", "matplotlib", "nbformat", "nbclient"]},
        "notebook_repository_commit": git_commit(REPO_ROOT),
        "source_commit_expected": CONFIG["expected_source_commit"],
        "source_repository_commit": git_commit(SOURCE_REPO),
        "protocol_version": CONFIG["protocol_version"],
        "profile": CONFIG["profile"],
        "benchmark_dir": str(BENCHMARK_DIR),
        "selected_job_id": SELECTED_JOB_ID,
        "selected_representations": [item["rep"] for item in SELECTED_REPS],
        "deterministic_regeneration": "simulate selected synthetic job from frozen job config only; no ScGeo methods or thresholds are changed",
        "threshold_policy": CONFIG["threshold_policy"],
    }
    if extra:
        metadata.update(extra)
    path = metadata_dir / f"{stem}_metadata.json"
    path.write_text(json.dumps(metadata, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return pd.DataFrame([{"metadata": rel_display(path)}])

In [ ]:
required_files = [
    job_file("config.json"),
    job_file("status.json"),
    job_file("runtime_summary.csv"),
    job_file("state_metrics.csv"),
    job_file("shift_estimate_diagnostics.csv"),
    job_file("representation_consensus.csv"),
    job_file("representation_quality_outlier.csv"),
    job_file("representation_distortion_detection.csv"),
    job_file("explicit_corruption_detection.csv"),
    job_file("neighborhood_discrimination.csv"),
    job_file("distorted_state_detection.csv"),
    job_file("alignment_accuracy.csv"),
    job_file("bootstrap_coverage.csv"),
    job_file("coverage_summary.csv"),
    job_file("summary_metrics.csv"),
    job_file("sample_center_diagnostics.csv"),
]
verified_files = verify_selected_frozen_files(required_files)
job_config = json.loads((BENCHMARK_DIR / job_file("config.json")).read_text(encoding="utf-8"))
status = json.loads((BENCHMARK_DIR / job_file("status.json")).read_text(encoding="utf-8"))
if job_config["seed_split"] != "evaluation":
    raise AssertionError("Selected job must be held-out evaluation")
if job_config["scenario"] != "representation_corruption":
    raise AssertionError("Selected job must be representation_corruption")
if status.get("status") != "completed":
    raise AssertionError("Selected frozen job is not completed")

pd.DataFrame([{
    "job_id": SELECTED_JOB_ID,
    "scenario": job_config["scenario"],
    "seed_split": job_config["seed_split"],
    "seed": job_config["seed"],
    "protocol": CONFIG["protocol_version"],
    "source_commit": CONFIG["expected_source_commit"],
    "verified_files": len(verified_files),
    "threshold_policy": CONFIG["threshold_policy"],
}])

## Deterministic geometry regeneration

The frozen CSV outputs do not store per-cell coordinates. For visualization only, this notebook reconstructs the selected synthetic job using the recorded frozen job configuration and the benchmark simulator source at the pinned commit. The regenerated cell counts are checked against the frozen per-sample state counts before any plotting.

In [ ]:
sim = load_simulation_module()
profile_cfg = job_config["profile_cfg"]
adata = sim.simulate_perturbation_geometry(
    scenario=job_config["scenario"],
    n_samples_per_condition=int(profile_cfg["n_samples_per_condition"]),
    cells_per_sample=int(profile_cfg["cells_per_sample"]),
    seed=int(job_config["seed"]),
    velocity_mode=job_config.get("velocity_mode"),
)
truth = adata.uns["simulation_truth"]
expected_reps = [item["rep"] for item in SELECTED_REPS]
missing_reps = sorted(set(expected_reps).difference(adata.obsm.keys()))
if missing_reps:
    raise AssertionError(f"Regenerated geometry is missing selected representations: {missing_reps}")

runtime = read_job_csv("runtime_summary.csv")
if int(runtime["n_obs"].iloc[0]) != int(adata.n_obs):
    raise AssertionError("Regenerated n_obs does not match frozen runtime_summary")

frozen_counts = read_job_csv("sample_center_diagnostics.csv")[["sample", "state", "n_cells"]].sort_values(["sample", "state"]).reset_index(drop=True)
regen_counts = (
    adata.obs.groupby(["sample", "state"], observed=False)
    .size()
    .rename("n_cells")
    .reset_index()
    .sort_values(["sample", "state"])
    .reset_index(drop=True)
)
pd.testing.assert_frame_equal(frozen_counts, regen_counts, check_dtype=False)

pd.DataFrame([{
    "regenerated_cells": int(adata.n_obs),
    "regenerated_representations": ",".join(sorted(adata.obsm.keys())),
    "count_check": "matches frozen sample_center_diagnostics",
    "velocity_mode": str(truth.get("velocity_mode")),
    "velocity_keys_present": truth.get("velocity_keys") is not None,
}])

In [ ]:
quality = read_job_csv("representation_quality_outlier.csv")
rep_distortion = read_job_csv("representation_distortion_detection.csv")
explicit_corruption = read_job_csv("explicit_corruption_detection.csv")
neighborhood = read_job_csv("neighborhood_discrimination.csv")
consensus = read_job_csv("representation_consensus.csv")
alignment = read_job_csv("alignment_accuracy.csv")
summary_metrics = read_job_csv("summary_metrics.csv")

selected_rep_df = pd.DataFrame(SELECTED_REPS)

neighbor_rows = []
for item in SELECTED_REPS:
    rep = item["rep"]
    row = {"rep": rep, "display": item["display"], "role": item["role"]}
    if rep == "X_truth":
        row.update({"reference_rep": "X_truth", "k": 30, "neighbor_overlap": 1.0, "neighbor_jaccard": 1.0, "pair_class": "reference"})
    else:
        pair = neighborhood[
            (((neighborhood["rep_a"] == "X_truth") & (neighborhood["rep_b"] == rep))
             | ((neighborhood["rep_b"] == "X_truth") & (neighborhood["rep_a"] == rep)))
            & (neighborhood["k"] == 30)
        ]
        values = pair.pivot_table(index=["rep_a", "rep_b", "k", "pair_class"], columns="metric", values="value", aggfunc="first").reset_index()
        if values.empty:
            row.update({"reference_rep": "X_truth", "k": 30, "neighbor_overlap": np.nan, "neighbor_jaccard": np.nan, "pair_class": "not_available"})
        else:
            values_row = values.iloc[0]
            row.update({
                "reference_rep": "X_truth",
                "k": int(values_row["k"]),
                "neighbor_overlap": float(values_row.get("neighbor_overlap", np.nan)),
                "neighbor_jaccard": float(values_row.get("neighbor_jaccard", np.nan)),
                "pair_class": values_row["pair_class"],
            })
    neighbor_rows.append(row)
neighbor_summary = pd.DataFrame(neighbor_rows)

quality_cols = ["rep", "representation_category", "neighborhood_outlier", "distortion_outlier", "state_graph_outlier", "predicted_quality_outlier", "status"]
rep_status = (
    selected_rep_df
    .merge(quality[quality_cols], on="rep", how="left")
    .merge(rep_distortion[["rep", "true_representation_distorted", "predicted_representation_distorted", "status"]].rename(columns={"status": "distortion_status"}), on="rep", how="left")
    .merge(explicit_corruption[["rep", "true_corrupted", "predicted_corrupted", "status"]].rename(columns={"status": "explicit_corruption_status"}), on="rep", how="left")
    .merge(neighbor_summary[["rep", "reference_rep", "k", "neighbor_overlap", "neighbor_jaccard", "pair_class"]], on="rep", how="left")
)
rep_status["velocity_alignment"] = "unavailable: velocity not requested in frozen job"
rep_status

In [ ]:
# Deterministic visual subset only; condition centers below are computed from all cells.
visual_rng = np.random.default_rng(20260717)
obs = adata.obs.reset_index(names="cell_id")
selected_indices = []
for (_, _), group in obs.groupby(["state", "condition"], observed=False):
    idx = group.index.to_numpy()
    take = min(160, len(idx))
    selected_indices.extend(visual_rng.choice(idx, size=take, replace=False).tolist())
selected_indices = sorted(selected_indices)
visual_obs = obs.loc[selected_indices].copy()

cell_rows = []
center_rows = []
for item in SELECTED_REPS:
    rep = item["rep"]
    coords = np.asarray(adata.obsm[rep])[:, :2]
    for row_idx, obs_row in visual_obs.iterrows():
        x, y = coords[row_idx]
        cell_rows.append({
            "row_type": "cell_visual_subset",
            "rep": rep,
            "display": item["display"],
            "role": item["role"],
            "cell_id": obs_row["cell_id"],
            "state": obs_row["state"],
            "condition": obs_row["condition"],
            "x": float(x),
            "y": float(y),
        })
    coord_df = pd.DataFrame(coords, columns=["x", "y"])
    coord_df["state"] = adata.obs["state"].to_numpy()
    coord_df["condition"] = adata.obs["condition"].to_numpy()
    centers = coord_df.groupby(["state", "condition"], observed=False, as_index=False).agg(x=("x", "mean"), y=("y", "mean"))
    for _, center_row in centers.iterrows():
        center_rows.append({
            "row_type": "condition_center_all_cells",
            "rep": rep,
            "display": item["display"],
            "role": item["role"],
            "cell_id": "",
            "state": center_row["state"],
            "condition": center_row["condition"],
            "x": float(center_row["x"]),
            "y": float(center_row["y"]),
        })

plot_source = pd.concat([pd.DataFrame(cell_rows), pd.DataFrame(center_rows)], ignore_index=True)
plot_source.head()

In [ ]:
consensus_counts = consensus["consensus_label"].value_counts().to_dict()
fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)
axes = axes.ravel()

for ax, item in zip(axes, SELECTED_REPS):
    rep = item["rep"]
    rep_cells = plot_source[(plot_source["rep"] == rep) & (plot_source["row_type"] == "cell_visual_subset")]
    rep_centers = plot_source[(plot_source["rep"] == rep) & (plot_source["row_type"] == "condition_center_all_cells")]
    status = rep_status[rep_status["rep"] == rep].iloc[0]
    for condition, marker in CONDITION_MARKERS.items():
        subset = rep_cells[rep_cells["condition"] == condition]
        for state, state_subset in subset.groupby("state", observed=False):
            ax.scatter(
                state_subset["x"],
                state_subset["y"],
                s=9,
                alpha=0.28 if condition == "control" else 0.40,
                marker=marker,
                color=STATE_COLORS.get(state, "#777777"),
                linewidths=0,
            )
    for state, state_centers in rep_centers.groupby("state", observed=False):
        wide = state_centers.set_index("condition")
        if {"control", "treated"}.issubset(wide.index):
            x0, y0 = wide.loc["control", ["x", "y"]]
            x1, y1 = wide.loc["treated", ["x", "y"]]
            ax.scatter([x0], [y0], marker="o", s=42, color=STATE_COLORS.get(state, "#777777"), edgecolor="black", linewidth=0.5, zorder=4)
            ax.scatter([x1], [y1], marker="^", s=50, color=STATE_COLORS.get(state, "#777777"), edgecolor="black", linewidth=0.5, zorder=4)
            ax.annotate("", xy=(x1, y1), xytext=(x0, y0), arrowprops={"arrowstyle": "->", "color": STATE_COLORS.get(state, "#777777"), "lw": 1.2, "alpha": 0.9})
            ax.text(x1, y1, state.replace("state_", "s"), fontsize=8, color="black", ha="left", va="bottom")
    summary_text = (
        f"category: {status['representation_category']}\n"
        f"quality status: {status['status']}; outlier={bool(status['predicted_quality_outlier'])}\n"
        f"neighbor Jaccard@30: {status['neighbor_jaccard']:.3f}\n"
        f"consensus: {consensus_counts}\n"
        f"velocity: unavailable"
    )
    ax.text(0.02, 0.98, summary_text, transform=ax.transAxes, ha="left", va="top", fontsize=8, bbox={"boxstyle": "round,pad=0.25", "fc": "white", "ec": "#cccccc", "alpha": 0.86})
    ax.set_title(f"{item['display']}\n{rep}")
    ax.set_xlabel("dimension 1")
    ax.set_ylabel("dimension 2")

legend_handles = []
for state, color in STATE_COLORS.items():
    legend_handles.append(plt.Line2D([0], [0], marker="o", color="none", markerfacecolor=color, label=state, markersize=6))
legend1 = axes[0].legend(handles=legend_handles, title="State", loc="lower left", frameon=True, fontsize=8)
axes[0].add_artist(legend1)
condition_handles = [
    plt.Line2D([0], [0], marker="o", color="black", label="control center", markersize=6, linestyle="none"),
    plt.Line2D([0], [0], marker="^", color="black", label="treated center", markersize=6, linestyle="none"),
]
axes[1].legend(handles=condition_handles, title="Condition centers", loc="lower left", frameon=True, fontsize=8)

alt = (
    "Geometry stress-test panel for held-out representation-corruption seed 5. "
    "The reference and equivalent rotated representations preserve neighborhoods, the anisotropic representation is diagnostically distorted, "
    "and the corrupted representation has near-zero neighborhood preservation. Arrows show control-to-treated condition-center displacement per state. "
    "Frozen consensus labels include three representation-unstable states and two stable-neutral states; velocity alignment is unavailable because the frozen job did not request velocity."
)
figure_artifacts = write_figure_bundle("04_synthetic_geometry_stress_test", fig, plot_source, alt)
figure_artifacts

In [ ]:
rep_summary_alt = (
    "Representation-level evidence table for the selected held-out geometry stress-test job. "
    "Reference and rotated representations are assessed as non-outlier equivalents; the anisotropic diagnostic representation is a quality and distortion outlier; "
    "the explicitly corrupted representation is correctly identified as corrupted while local-distortion status is not applicable for that corruption category."
)
write_table_artifact("04_representation_neighborhood_quality_summary", rep_status, rep_summary_alt)
rep_status

In [ ]:
state_metrics = read_job_csv("state_metrics.csv")
robust_effect = state_metrics[state_metrics["method"] == "robust_geometric_median"].copy()
shift_diag = read_job_csv("shift_estimate_diagnostics.csv")
bootstrap = read_job_csv("bootstrap_coverage.csv")
distorted = read_job_csv("distorted_state_detection.csv")

local_diag = (
    distorted[(distorted["rep_a"] == "X_truth") & (distorted["rep_b"] == "X_anisotropic")]
    .groupby("state", as_index=False)
    .agg(
        diagnostic_local_score_max=("local_distortion_score", "max"),
        diagnostic_local_threshold=("threshold", "first"),
        diagnostic_true_local_distortion=("true_representation_local_distortion", "max"),
        diagnostic_predicted_local_distortion=("predicted_representation_local_distortion", "max"),
        diagnostic_local_status=("status", "first"),
    )
)
local_corrupt = (
    distorted[(distorted["rep_a"] == "X_truth") & (distorted["rep_b"] == "X_corrupted")]
    .groupby("state", as_index=False)
    .agg(
        corrupted_pair_score_max=("local_distortion_score", "max"),
        corrupted_pair_predicted_distorted=("predicted_representation_local_distortion", "max"),
        corrupted_pair_status=("status", "first"),
    )
)

consensus_evidence = consensus[["state", "consensus_label", "status", "n_usable_representations", "usable_fraction", "loo_rep_magnitude_max_relative_deviation", "velocity_requested"]].copy()
alignment_consensus = alignment[alignment["source"] == "consensus"][["state", "predicted_class", "velocity_requested"]].rename(columns={"predicted_class": "dynamics_predicted_class"})

evidence = (
    robust_effect[["state", "estimated_magnitude", "true_magnitude", "predicted_shifted", "true_shifted", "threshold"]]
    .merge(shift_diag[["state", "delta_norm", "normalized_delta_norm", "bootstrap_ci95_low", "bootstrap_ci95_high", "direction_stability", "sign_stability"]], on="state", how="left")
    .merge(consensus_evidence, on="state", how="left")
    .merge(local_diag, on="state", how="left")
    .merge(local_corrupt, on="state", how="left")
    .merge(alignment_consensus, on="state", how="left")
    .merge(bootstrap[["state", "coverage_applicable", "coverage_status", "covered"]], on="state", how="left")
)

def robust_effect_label(row):
    return (
        f"pred={bool(row['predicted_shifted'])}; true={bool(row['true_shifted'])}; "
        f"delta={row['delta_norm']:.3f}; CI=[{row['bootstrap_ci95_low']:.3f},{row['bootstrap_ci95_high']:.3f}]"
    )

def local_geometry_label(row):
    return (
        f"diagnostic true={bool(row['diagnostic_true_local_distortion'])}/pred={bool(row['diagnostic_predicted_local_distortion'])}; "
        f"score={row['diagnostic_local_score_max']:.3f} < threshold={row['diagnostic_local_threshold']:.3f}; "
        f"corrupted pair status={row['corrupted_pair_status']}"
    )

def dynamics_label(row):
    if bool(row.get("velocity_requested", False)):
        return str(row.get("dynamics_predicted_class", "unavailable"))
    return "unavailable: velocity not requested in frozen job"

def coverage_reason(row):
    if bool(row["coverage_applicable"]):
        return f"bootstrap {row['coverage_status']}; covered={row['covered']}"
    return f"bootstrap {row['coverage_status']}; zero-effect state not a magnitude-coverage target"

evidence["robust_effect"] = evidence.apply(robust_effect_label, axis=1)
evidence["representation_stability"] = evidence.apply(lambda row: f"{row['consensus_label']} ({row['status']})", axis=1)
evidence["local_geometry_status"] = evidence.apply(local_geometry_label, axis=1)
evidence["dynamics_status"] = evidence.apply(dynamics_label, axis=1)
evidence["coverage_status_reason"] = evidence.apply(coverage_reason, axis=1)

evidence_table = evidence[[
    "state",
    "robust_effect",
    "representation_stability",
    "local_geometry_status",
    "dynamics_status",
    "coverage_status_reason",
    "estimated_magnitude",
    "true_magnitude",
    "threshold",
    "normalized_delta_norm",
    "loo_rep_magnitude_max_relative_deviation",
    "diagnostic_true_local_distortion",
    "diagnostic_predicted_local_distortion",
    "corrupted_pair_status",
]].sort_values("state")

evidence_alt = (
    "Per-state evidence table for the selected held-out stress-test job. Robust effects are shown together with representation stability, local-geometry status, dynamics availability, and coverage reason. "
    "The table keeps negative evidence visible: states 0 and 1 have true diagnostic local distortion but no predicted local-distortion call, and velocity alignment is unavailable for every state in this frozen job."
)
write_table_artifact("04_state_evidence_table", evidence_table, evidence_alt)
evidence_table

In [ ]:
negative_summary = pd.DataFrame([
    {
        "negative_result": "state-level representation instability is visible",
        "evidence": ", ".join(consensus.loc[consensus["consensus_label"] == "representation_unstable", "state"].tolist()),
    },
    {
        "negative_result": "diagnostic local-distortion recall is incomplete in this job",
        "evidence": "X_truth vs X_anisotropic has true local distortion for state_0 and state_1, but frozen predicted local-distortion calls are false",
    },
    {
        "negative_result": "explicit corruption is a representation-level finding, not a local-distortion success",
        "evidence": "X_corrupted is predicted corrupted; pair/state local-distortion rows are not_applicable for corrupted pairs",
    },
    {
        "negative_result": "velocity alignment is unavailable for this selected job",
        "evidence": "frozen alignment_accuracy has velocity_requested=False and predicted_class=unavailable for every state",
    },
])
write_table_artifact("04_negative_results_visible", negative_summary)
metadata = write_metadata("04_synthetic_geometry_stress_test", {
    "figures": ["04_synthetic_geometry_stress_test"],
    "verified_frozen_files": int(len(verified_files)),
    "regenerated_cells": int(adata.n_obs),
    "visual_subset_cells_per_state_condition_max": 160,
    "new_thresholds_tuned": False,
    "scgeo_methods_changed": False,
})
metadata